# Solution F3 · Mean versus median: OLS and quantile regression

**Type**: Compare

## Objective

Compare `LinearRegression` (which estimates the conditional mean) and `QuantileRegressor(quantile=0.5)` (which estimates the
conditional median) on skewed and heavy-tailed noise, and see that each one wins on the metric it optimizes.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, QuantileRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error


def generate(n, seed, noise="lognormal"):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 10, n)
    eps = rng.lognormal(0, 1, n) if noise == "lognormal" else rng.standard_t(2, n)
    return x.reshape(-1, 1), 5 + 2 * x + eps, eps

## Commented solution

In [ ]:
X_tr, y_tr, eps_tr = generate(500, 0)
ols = LinearRegression().fit(X_tr, y_tr)
median = QuantileRegressor(quantile=0.5, alpha=0, solver="highs").fit(X_tr, y_tr)
ols_intercept, med_intercept = ols.intercept_, median.intercept_
ols_slope, med_slope = ols.coef_[0], median.coef_[0]
print(f"slopes    : OLS {ols_slope:.3f} | median {med_slope:.3f}   (true: 2)")
print(f"intercepts: OLS {ols_intercept:.3f} | median {med_intercept:.3f}   (true: 5 + mean or median of the noise)")

In [ ]:
theoretical_gap = eps_tr.mean() - np.median(eps_tr)
intercept_gap = ols_intercept - med_intercept
print(f"mean - median of the noise: {theoretical_gap:.3f} | difference between the two intercepts: {intercept_gap:.3f}")

The two lines have the **same slope** (the noise does not depend on x) but different intercepts: OLS passes through the center
of mass of the cloud (5 + mean of the noise), median regression splits it in half (5 + median). With skewed
noise mean and median do not coincide.

In [ ]:
X_te, y_te, _ = generate(5000, 1)
p_ols, p_med = ols.predict(X_te), median.predict(X_te)
mae_ols, mae_med = mean_absolute_error(y_te, p_ols), mean_absolute_error(y_te, p_med)
rmse_ols = mean_squared_error(y_te, p_ols, squared=False)
rmse_med = mean_squared_error(y_te, p_med, squared=False)
print(f"MAE : OLS {mae_ols:.3f} | median {mae_med:.3f}")
print(f"RMSE: OLS {rmse_ols:.3f} | median {rmse_med:.3f}")

In [ ]:
slopes_ols, slopes_med = [], []
for s in range(200):
    X, y, _ = generate(100, 100 + s, "t")
    slopes_ols.append(LinearRegression().fit(X, y).coef_[0])
    slopes_med.append(QuantileRegressor(quantile=0.5, alpha=0, solver="highs").fit(X, y).coef_[0])
std_slope_ols = np.std(slopes_ols, ddof=1)
std_slope_med = np.std(slopes_med, ddof=1)
print(f"standard deviation of the slope estimated over 200 samples: OLS {std_slope_ols:.3f} | median {std_slope_med:.3f}")

In [ ]:
true_false = {"a": True, "b": False, "c": True}

## Check

In [ ]:
with verify():
    check(abs(ols_slope - 2) < 0.15 and abs(med_slope - 2) < 0.15,
          "the two slopes are both close to the true value 2 (the noise is independent of x)",
          "both slopes should be about 2")
    check(abs(intercept_gap - theoretical_gap) < 0.4 and intercept_gap > 0.2,
          "the two intercepts differ by about mean - median of the noise",
          "OLS and median regression must have intercepts that differ by about (mean - median) of the noise")
    check(mae_med < mae_ols, "median regression wins on the MAE", "on the MAE median regression should win")
    check(rmse_ols < rmse_med, "OLS wins on the RMSE", "on the RMSE OLS should win")
    check(len(slopes_ols) == 200 and len(slopes_med) == 200, "200 estimates per method", "you need 200 slopes for each method")
    check(std_slope_ols > 1.3 * std_slope_med, "with heavy tails the OLS estimate of the slope is much less stable",
          "with t(2) noise the standard deviation of OLS should exceed by at least 30% that of median regression")
    check(true_false == {"a": True, "b": False, "c": True}, "statements judged correctly",
          "the two lines differ in the intercept, not in the slope")
done()

## What you learned

- **The loss decides what you estimate.** Minimizing the squared error leads to the conditional mean; minimizing the absolute
  error leads to the conditional median. Each model wins on its own metric: the median on the MAE, OLS on the RMSE.
  There is no "best" without saying which error costs.
- With skewed noise the two estimates differ in the intercept, that is in the central tendency of the distribution
  of the noise; the slopes coincide.
- With heavy tails (the t with 2 degrees of freedom has infinite variance) OLS becomes unstable: the few enormous values
  dominate the sum of squares. The slope estimate has a spread almost double that of the median one. It is the same
  reason why Huber and RANSAC (exercise C2) cope better with outliers.

## Common pitfall

Choosing the model by looking at the metric that the model optimizes. If the model minimizes the square of the
errors, it almost certainly has the best RMSE: the comparison is rigged. The comparison metric must come from the
cost of the error for the problem, and the model must be trained with the loss that reflects it.

## Interview question

> *Click on a question to show its answer.*

<details><summary>You are predicting delivery times. The distribution is skewed, with long tails. Linear regression or quantile regression?</summary>

It depends on what I have to tell the customer. Linear regression estimates the mean time, pulled up by the tail: a number
that is usually higher than what happens in most deliveries. The median describes the typical case better,
and it is robust to extreme values. If what matters to the company is not breaking a promise, I would use a high quantile (for
example the 90th percentile), which quantile regression estimates directly. I would choose the loss based on the cost of the error.

</details>